# ⚡ Colab GPU Training Tunnel — Vietnamese Legal Reranker (LoRA)

**Hướng dẫn (chỉ 1 bước):**
1. Trên menu Colab chọn **Runtime $\rightarrow$ Change runtime type $\rightarrow$ T4 GPU**.
2. Bấm **Run (▶)** ở ô code duy nhất bên dưới, đợi ~30 giây đến khi hiện đường link `https://xxxx.trycloudflare.com`.
3. Copy link đó và chạy lệnh sau trên **PowerShell ở máy Local của bạn**:
```powershell
.\.venv\Scripts\python.exe scripts\training\train_reranker_tunnel.py --colab-url https://xxxx.trycloudflare.com
```
Toàn bộ dữ liệu và code điều khiển vẫn nằm ở máy Local của bạn; Colab chỉ làm GPU Worker huấn luyện và trả trọng số `lora_reranker_weights.pt` về máy bạn.

In [ ]:
# === Ô CODE DUY NHẤT: CÀI ĐẶT, KHỞI CHẠY GPU TRAINING WORKER & MỞ CLOUDFLARE TUNNEL ===
import subprocess, time, re, pathlib

print('1/3 Đang cài đặt fastapi, uvicorn và tải cloudflared...')
subprocess.run(['pip', 'install', '-q', 'fastapi', 'uvicorn', 'transformers', 'accelerate'], check=True)
subprocess.run('wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared && chmod +x /usr/local/bin/cloudflared', shell=True, check=True)

WORKER_CODE = '"""GPU Training Worker chạy trên Google Colab (kết nối với máy Local qua Cloudflare Tunnel).\n\nChức năng:\n1. Mở HTTP endpoints (`/health`, `/train_stream`, `/download_weights`).\n2. Nhận dữ liệu `reranker_train_groups.jsonl` (1,401 nhóm Hard Negatives) và\n   `reranker_val_candidates.jsonl` (351 câu Validation) gửi trực tiếp từ máy Local.\n3. Gắn LoRA (`r=16, alpha=32`) vào toàn bộ các tầng Linear của `BAAI/bge-reranker-base`\n   cùng tầng `classifier`, huấn luyện với hàm mất mát kết hợp:\n   - ListNet / InfoNCE Cross-Entropy Loss trên nhóm (1 Positive vs K Hard Negatives)\n   - Pairwise Margin Ranking Loss (ép điểm Positive > Hard Negative tối thiểu một khoảng margin)\n4. Đánh giá trên 351 câu Validation sau mỗi epoch, tối ưu hệ số `fusion_alpha` và `pool_size`,\n   stream log thời gian thực về Terminal máy Local, và trả file trọng số `lora_reranker_weights.pt`\n   (~7 MB) về lưu tại `data/indices/reranker_finetuned/lora_reranker_weights.pt` trên máy Local.\n"""\n\nfrom __future__ import annotations\n\nimport gzip\nimport io\nimport json\nimport math\nfrom pathlib import Path\nimport random\nimport time\nfrom typing import Any, Dict, Generator, List, Tuple\n\nfrom fastapi import FastAPI, Request\nfrom fastapi.responses import FileResponse, JSONResponse, StreamingResponse\nimport torch\nimport torch.nn as nn\nimport torch.nn.functional as F\nfrom transformers import AutoModelForSequenceClassification, AutoTokenizer, get_cosine_schedule_with_warmup\n\napp = FastAPI(title="Vietnamese Legal Reranker Colab GPU Trainer")\n\nBASE_MODEL_NAME = "BAAI/bge-reranker-base"\nWEIGHTS_OUT_PATH = Path("/tmp/lora_reranker_weights.pt")\n\n\nclass LoRALinear(nn.Module):\n    """Lớp bọc LoRA thuần PyTorch cho nn.Linear (không phụ thuộc phiên bản peft)."""\n\n    def __init__(self, base_layer: nn.Linear, r: int = 16, alpha: float = 32.0, dropout: float = 0.05) -> None:\n        super().__init__()\n        self.base_layer = base_layer\n        for p in self.base_layer.parameters():\n            p.requires_grad = False\n\n        in_features = base_layer.in_features\n        out_features = base_layer.out_features\n        self.r = r\n        self.alpha = alpha\n        self.scaling = alpha / float(r)\n        self.dropout = nn.Dropout(dropout)\n\n        device = base_layer.weight.device\n        self.lora_A = nn.Parameter(torch.empty((r, in_features), device=device, dtype=torch.float32))\n        self.lora_B = nn.Parameter(torch.zeros((out_features, r), device=device, dtype=torch.float32))\n        nn.init.kaiming_uniform_(self.lora_A, a=math.sqrt(5))\n\n    def forward(self, x: torch.Tensor) -> torch.Tensor:\n        base_out = self.base_layer(x)\n        lora_out = (self.dropout(x.float()) @ self.lora_A.t() @ self.lora_B.t()) * self.scaling\n        return base_out + lora_out.to(base_out.dtype)\n\n\ndef inject_lora(model: nn.Module, r: int = 16, alpha: float = 32.0) -> List[str]:\n    """Gắn LoRA vào các tầng attention & intermediate/output dense của RoBERTa."""\n    target_suffixes = (\n        "attention.self.query",\n        "attention.self.key",\n        "attention.self.value",\n        "attention.output.dense",\n        "intermediate.dense",\n        "output.dense",\n    )\n    replaced: List[str] = []\n    for p in model.parameters():\n        p.requires_grad = False\n\n    for name, module in list(model.named_modules()):\n        if not name.startswith("roberta.encoder.layer."):\n            continue\n        if isinstance(module, nn.Linear) and name.endswith(target_suffixes):\n            parent_name, child_name = name.rsplit(".", 1)\n            parent = model.get_submodule(parent_name)\n            setattr(parent, child_name, LoRALinear(module, r=r, alpha=alpha))\n            replaced.append(name)\n\n    # Mở khóa tầng classifier head để học ranh giới quyết định cho văn bản luật\n    for p in model.classifier.parameters():\n        p.requires_grad = True\n\n    return replaced\n\n\ndef extract_lora_and_classifier_state(\n    model: nn.Module,\n) -> Tuple[Dict[str, torch.Tensor], Dict[str, torch.Tensor]]:\n    lora_state: Dict[str, torch.Tensor] = {}\n    for name, module in model.named_modules():\n        if isinstance(module, LoRALinear):\n            lora_state[f"{name}.lora_A"] = module.lora_A.detach().cpu().half()\n            lora_state[f"{name}.lora_B"] = module.lora_B.detach().cpu().half()\n\n    cls_state: Dict[str, torch.Tensor] = {}\n    for k, v in model.classifier.state_dict().items():\n        # Trường hợp bên trong classifier có tầng nào được bọc hoặc giữ nguyên\n        cls_state[k] = v.detach().cpu()\n    return lora_state, cls_state\n\n\ndef compute_ndcg_at_10(retrieved_ids: List[int], relevant_set: set[int]) -> float:\n    if not relevant_set:\n        return 0.0\n    dcg = 0.0\n    for rank_idx, doc_id in enumerate(retrieved_ids[:10], start=1):\n        if doc_id in relevant_set:\n            dcg += 1.0 / math.log2(rank_idx + 1)\n    ideal_hits = min(len(relevant_set), 10)\n    idcg = sum(1.0 / math.log2(r + 1) for r in range(1, ideal_hits + 1))\n    return dcg / idcg if idcg > 0 else 0.0\n\n\ndef compute_mrr_at_10(retrieved_ids: List[int], relevant_set: set[int]) -> float:\n    for rank_idx, doc_id in enumerate(retrieved_ids[:10], start=1):\n        if doc_id in relevant_set:\n            return 1.0 / rank_idx\n    return 0.0\n\n\n@torch.inference_mode()\ndef evaluate_validation(\n    model: nn.Module,\n    tokenizer: Any,\n    val_records: List[Dict[str, Any]],\n    device: str,\n    max_length: int = 384,\n) -> Dict[str, Any]:\n    model.eval()\n    all_query_ce_probs: List[List[float]] = []\n\n    for rec in val_records:\n        qtext = rec["query_text"]\n        cands = rec["candidates"]\n        texts = [c["text"] for c in cands]\n        queries = [qtext] * len(texts)\n        enc = tokenizer(\n            queries,\n            texts,\n            padding=True,\n            truncation=True,\n            max_length=max_length,\n            return_tensors="pt",\n        ).to(device)\n        with torch.cuda.amp.autocast(enabled=("cuda" in device)):\n            logits = model(**enc).logits.squeeze(-1)\n            if logits.ndim == 0:\n                logits = logits.unsqueeze(0)\n        probs = torch.sigmoid(logits.float()).cpu().tolist()\n        all_query_ce_probs.append(probs)\n\n    best_cfg = {"ndcg10": -1.0, "mrr10": 0.0, "hit1": 0.0, "hit3": 0.0, "alpha": 0.6, "pool_size": 10}\n\n    for pool_size in (10, 15, 20):\n        for alpha in (0.25, 0.35, 0.45, 0.55, 0.65, 0.75, 0.85, 1.0):\n            ndcg_sum = mrr_sum = hit1_sum = hit3_sum = 0.0\n            for rec, ce_probs in zip(val_records, all_query_ce_probs):\n                rel_set = set(rec["relevant_article_ids"])\n                cands = rec["candidates"]\n                pk = min(pool_size, len(cands))\n\n                sub_ce = ce_probs[:pk]\n                c_min, c_max = min(sub_ce), max(sub_ce)\n                c_span = (c_max - c_min) if (c_max - c_min) > 1e-8 else 1.0\n                ce_norm = [(x - c_min) / c_span for x in sub_ce]\n\n                sub_pr = [float(c["hybrid_score"]) for c in cands[:pk]]\n                p_min, p_max = min(sub_pr), max(sub_pr)\n                p_span = (p_max - p_min) if (p_max - p_min) > 1e-8 else 1.0\n                pr_norm = [(x - p_min) / p_span for x in sub_pr]\n\n                scored = [\n                    (cands[i]["article_id"], alpha * ce_norm[i] + (1.0 - alpha) * pr_norm[i])\n                    for i in range(pk)\n                ]\n                scored.sort(key=lambda x: x[1], reverse=True)\n                ranked_ids = [aid for aid, _ in scored] + [c["article_id"] for c in cands[pk:]]\n\n                hit1_sum += float(bool(set(ranked_ids[:1]) & rel_set))\n                hit3_sum += float(bool(set(ranked_ids[:3]) & rel_set))\n                ndcg_sum += compute_ndcg_at_10(ranked_ids, rel_set)\n                mrr_sum += compute_mrr_at_10(ranked_ids, rel_set)\n\n            n = len(val_records)\n            ndcg10 = ndcg_sum / n\n            if ndcg10 > best_cfg["ndcg10"]:\n                best_cfg = {\n                    "ndcg10": round(ndcg10, 4),\n                    "mrr10": round(mrr_sum / n, 4),\n                    "hit1": round(hit1_sum / n * 100, 2),\n                    "hit3": round(hit3_sum / n * 100, 2),\n                    "alpha": alpha,\n                    "pool_size": pool_size,\n                }\n\n    return best_cfg\n\n\n@app.get("/health")\ndef health() -> JSONResponse:\n    has_cuda = torch.cuda.is_available()\n    gpu_name = torch.cuda.get_device_name(0) if has_cuda else "CPU"\n    vram_gb = (\n        round(torch.cuda.get_device_properties(0).total_memory / (1024**3), 2)\n        if has_cuda\n        else 0.0\n    )\n    return JSONResponse(\n        {\n            "status": "ok",\n            "cuda_available": has_cuda,\n            "gpu_name": gpu_name,\n            "vram_gb": vram_gb,\n            "torch_version": torch.__version__,\n        }\n    )\n\n\n@app.post("/train_stream")\nasync def train_stream(request: Request) -> StreamingResponse:\n    raw_body = await request.body()\n    payload = json.loads(gzip.decompress(raw_body).decode("utf-8"))\n\n    train_groups: List[Dict[str, Any]] = payload["train_groups"]\n    val_records: List[Dict[str, Any]] = payload["val_records"]\n    cfg: Dict[str, Any] = payload.get("config", {})\n\n    epochs = int(cfg.get("epochs", 3))\n    lr = float(cfg.get("lr", 3e-4))\n    lora_r = int(cfg.get("lora_r", 16))\n    lora_alpha = float(cfg.get("lora_alpha", 32.0))\n    max_length = int(cfg.get("max_length", 384))\n    group_negs = int(cfg.get("group_negs", 4))\n    grad_accum = int(cfg.get("grad_accum", 4))\n    margin = float(cfg.get("margin", 1.0))\n\n    def event_stream() -> Generator[bytes, None, None]:\n        def emit(msg_type: str, **kwargs: Any) -> bytes:\n            data = {"type": msg_type, "ts": round(time.time(), 2), **kwargs}\n            return (json.dumps(data, ensure_ascii=False) + "\\n").encode("utf-8")\n\n        device = "cuda:0" if torch.cuda.is_available() else "cpu"\n        gpu_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"\n        yield emit("info", message=f"Bắt đầu trên thiết bị: {gpu_name} ({device})")\n\n        tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_NAME)\n        model = AutoModelForSequenceClassification.from_pretrained(BASE_MODEL_NAME)\n\n        # Đo điểm Zero-shot Pretrained trên tập Val (351 câu) trước khi train\n        model.to(device)\n        zs_val = evaluate_validation(model, tokenizer, val_records, device, max_length=max_length)\n        yield emit(\n            "val_zero_shot",\n            message=(\n                f"[Trước khi Fine-tune - Zero-shot] Val nDCG@10={zs_val[\'ndcg10\']} | "\n                f"Hit@1={zs_val[\'hit1\']}% | Hit@3={zs_val[\'hit3\']}%"\n            ),\n            metrics=zs_val,\n        )\n\n        replaced_layers = inject_lora(model, r=lora_r, alpha=lora_alpha)\n        model.to(device)\n        trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)\n        total_params = sum(p.numel() for p in model.parameters())\n        yield emit(\n            "info",\n            message=(\n                f"Đã gắn LoRA (r={lora_r}, alpha={lora_alpha}) vào {len(replaced_layers)} tầng Linear + Classifier. "\n                f"Tham số huấn luyện: {trainable_params:,} / {total_params:,} ({trainable_params/total_params*100:.2f}%)"\n            ),\n        )\n\n        optimizer = torch.optim.AdamW(\n            [p for p in model.parameters() if p.requires_grad],\n            lr=lr,\n            weight_decay=0.01,\n        )\n        total_steps = (len(train_groups) * epochs) // grad_accum\n        scheduler = get_cosine_schedule_with_warmup(\n            optimizer,\n            num_warmup_steps=max(10, int(total_steps * 0.1)),\n            num_training_steps=max(20, total_steps),\n        )\n        scaler = torch.cuda.amp.GradScaler(enabled=("cuda" in device))\n\n        best_val = zs_val.copy()\n        best_lora_state, best_cls_state = extract_lora_and_classifier_state(model)\n\n        rng = random.Random(42)\n\n        for epoch in range(1, epochs + 1):\n            model.train()\n            shuffled = list(train_groups)\n            rng.shuffle(shuffled)\n            running_loss = 0.0\n            step_count = 0\n            optimizer.zero_grad()\n            t_ep = time.time()\n\n            for idx, grp in enumerate(shuffled, start=1):\n                qtext = grp["query_text"]\n                pos_item = rng.choice(grp["positives"])\n                negs = grp["hard_negatives"]\n                if len(negs) > group_negs:\n                    # Ưu tiên lấy các hard negative ở thứ hạng cao nhất + lấy mẫu ngẫu nhiên\n                    sampled_negs = negs[:2] + rng.sample(negs[2:], group_negs - 2)\n                else:\n                    sampled_negs = list(negs)\n\n                texts = [pos_item["text"]] + [n["text"] for n in sampled_negs]\n                queries = [qtext] * len(texts)\n\n                enc = tokenizer(\n                    queries,\n                    texts,\n                    padding=True,\n                    truncation=True,\n                    max_length=max_length,\n                    return_tensors="pt",\n                ).to(device)\n\n                with torch.cuda.amp.autocast(enabled=("cuda" in device)):\n                    logits = model(**enc).logits.squeeze(-1)\n                    # 1. ListNet Cross-Entropy: ép xác suất của ứng viên Positive (index 0) tiến tới 1\n                    target = torch.zeros(1, dtype=torch.long, device=device)\n                    ce_loss = F.cross_entropy(logits.unsqueeze(0), target)\n                    # 2. Pairwise Margin Loss: ép logit(pos) - logit(neg_i) >= margin\n                    pos_logit = logits[0]\n                    neg_logits = logits[1:]\n                    margin_loss = F.relu(margin - (pos_logit - neg_logits)).mean()\n                    loss = (ce_loss + 0.5 * margin_loss) / grad_accum\n\n                scaler.scale(loss).backward()\n                running_loss += float(loss.item()) * grad_accum\n                step_count += 1\n\n                if idx % grad_accum == 0 or idx == len(shuffled):\n                    scaler.unscale_(optimizer)\n                    torch.nn.utils.clip_grad_norm_(\n                        [p for p in model.parameters() if p.requires_grad], 1.0\n                    )\n                    scaler.step(optimizer)\n                    scaler.update()\n                    optimizer.zero_grad()\n                    scheduler.step()\n\n                if idx % 200 == 0 or idx == len(shuffled):\n                    avg_l = running_loss / max(1, step_count)\n                    yield emit(\n                        "progress",\n                        epoch=epoch,\n                        step=idx,\n                        total=len(shuffled),\n                        loss=round(avg_l, 4),\n                        elapsed_sec=round(time.time() - t_ep, 1),\n                    )\n\n            val_metrics = evaluate_validation(\n                model, tokenizer, val_records, device, max_length=max_length\n            )\n            improved = val_metrics["ndcg10"] >= best_val["ndcg10"]\n            if improved:\n                best_val = val_metrics\n                best_lora_state, best_cls_state = extract_lora_and_classifier_state(model)\n\n            yield emit(\n                "epoch_done",\n                epoch=epoch,\n                val_metrics=val_metrics,\n                is_best=improved,\n                message=(\n                    f"Epoch {epoch}/{epochs} hoàn tất | Val nDCG@10={val_metrics[\'ndcg10\']} "\n                    f"| MRR@10={val_metrics[\'mrr10\']} | Hit@1={val_metrics[\'hit1\']}% "\n                    f"| Hit@3={val_metrics[\'hit3\']}% (alpha={val_metrics[\'alpha\']}, pool={val_metrics[\'pool_size\']})"\n                    + (" [NEW BEST ✅]" if improved else "")\n                ),\n            )\n\n        ckpt_obj = {\n            "base_model_name": BASE_MODEL_NAME,\n            "lora_r": lora_r,\n            "lora_alpha": lora_alpha,\n            "best_fusion_alpha": best_val["alpha"],\n            "best_pool_size": best_val["pool_size"],\n            "lora_state_dict": best_lora_state,\n            "classifier_state_dict": best_cls_state,\n            "metadata": {\n                "trained_at": time.strftime("%Y-%m-%d %H:%M:%S"),\n                "gpu_name": gpu_name,\n                "epochs": epochs,\n                "lr": lr,\n                "train_queries": len(train_groups),\n                "val_queries": len(val_records),\n                "val_zero_shot": zs_val,\n                "val_best_finetuned": best_val,\n            },\n        }\n        torch.save(ckpt_obj, WEIGHTS_OUT_PATH)\n        size_mb = round(WEIGHTS_OUT_PATH.stat().st_size / (1024 * 1024), 2)\n        yield emit(\n            "completed",\n            weights_size_mb=size_mb,\n            best_val=best_val,\n            zero_shot_val=zs_val,\n            message=f"Huấn luyện hoàn tất! Trọng số đã sẵn sàng tải về ({size_mb} MB).",\n        )\n\n    return StreamingResponse(event_stream(), media_type="application/x-ndjson")\n\n\n@app.get("/download_weights")\ndef download_weights() -> FileResponse:\n    if not WEIGHTS_OUT_PATH.exists():\n        raise RuntimeError("Chưa có file trọng số.")\n    return FileResponse(\n        path=str(WEIGHTS_OUT_PATH),\n        filename="lora_reranker_weights.pt",\n        media_type="application/octet-stream",\n    )\n'
pathlib.Path('/content/colab_gpu_worker.py').write_text(WORKER_CODE, encoding='utf-8')

print('2/3 Đang khởi động FastAPI GPU Worker tại cổng 8000...')
uvicorn_proc = subprocess.Popen(['uvicorn', 'colab_gpu_worker:app', '--host', '0.0.0.0', '--port', '8000'], cwd='/content')
time.sleep(3)

print('3/3 Đang mở đường hầm Cloudflare Tunnel...')
cf_proc = subprocess.Popen(['/usr/local/bin/cloudflared', 'tunnel', '--url', 'http://localhost:8000'], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
tunnel_url = None
for line in cf_proc.stdout:
    m = re.search(r'https://[a-zA-Z0-9-]+\.trycloudflare\.com', line)
    if m:
        tunnel_url = m.group(0)
        print('=' * 80)
        print('✅ COLAB GPU TUNNEL ĐÃ SẴN SÀNG! HÃY COPY URL DƯỚI ĐÂY:')
        print(f'\n    {tunnel_url}\n')
        print('👉 Chạy lệnh này trên PowerShell máy Local của bạn:')
        print(f'.\\.venv\\Scripts\\python.exe scripts\\training\\train_reranker_tunnel.py --colab-url {tunnel_url}')
        print('=' * 80)
        break

# Giữ cho tiến trình sống trong suốt lúc máy Local gửi lệnh train
try:
    for line in cf_proc.stdout:
        pass
except KeyboardInterrupt:
    print('Đã dừng GPU Tunnel.')
